# 12 · Scheduling, with Airflow

Every notebook so far ended with **you** typing a command. In production nobody
is awake to type it.

This notebook is one straight line:

1. write a tiny pipeline, and run it yourself
2. write a DAG that says when to run it
3. put the DAG where Airflow can see it
4. watch Airflow run it once
5. turn the schedule on, and watch it run **by itself**
6. turn it off again

**One pipeline file. One DAG file. Nothing else.**

## What Airflow is, in one sentence

> **Airflow is a program whose only job is to start other programs, in the right
> order, at the right time, and to remember what happened.**

It does not move data. It never touches a row. Every pipeline in this course
would run perfectly well if you typed the commands by hand at the right moments,
forever, without sleeping.

**Airflow is the thing that does the typing.**

## Why not just a for loop?

`cli.py` already runs the eight pipelines in order, and on a laptop that is
genuinely enough. It stops being enough the moment you ask any of these.

![](img/airflow-1-loop.png)

| The question | What a for loop answers |
|---|---|
| The third one failed at 3am. Did the rest still run? | *no, I stopped* |
| I want to retry just that one. How? | *rerun everything* |
| Six of these are independent. Why are they queued? | *because I am a loop* |
| Yesterday's file arrived late. Can I rerun just yesterday? | *no* |
| Who was told, and about which task? | *nobody, and nothing* |

Airflow answers all five. **That is the entire reason it exists.**

---

## Before anything: where the files live

This is the part that confuses people, so deal with it first.

![](img/airflow-4-where.png)

**Two files, both in this repo, doing different jobs.** The pipeline is the
work. The DAG says when. Both folders are mounted into the Airflow container,
which is why there is nothing to copy anywhere.

Airflow is already running in Docker, so one helper lets every cell below be a
single line.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import sql, fetch, run           # the same helpers as every other notebook

import subprocess, json, time, pathlib

PROJECT = pathlib.Path('..').resolve()        # this repo
DAGS    = PROJECT / 'airflow' / 'dags'       # the folder Airflow watches

CONTAINER = 'nightshift-airflow'

def airflow(*args, quiet=False):
    """Run an airflow command inside the container. The same CLI you would type
    after ssh-ing onto a scheduler box. Nothing here is special to notebooks."""
    r = subprocess.run(['docker', 'exec', CONTAINER, 'airflow', *args],
                       capture_output=True, text=True)
    out = (r.stdout + r.stderr).rstrip()
    if not quiet:
        print(out)
    return out

print('project  ', PROJECT)
print('dags     ', DAGS)
print('airflow  ', airflow('version', quiet=True))

---

# Step 1 · The work

A deliberately tiny pipeline. It writes one row saying *"I ran, at this time,
for this scheduled slot"*. That is enough to prove a schedule is live, and small
enough to read in one screen.

In [ ]:
HEARTBEAT = '''
from __future__ import annotations
import os, sys
import psycopg
from .lib.config import SCHEMA, dsn

DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.heartbeat (
    ran_at  TIMESTAMPTZ DEFAULT now(),
    slot    TEXT,            -- which scheduled minute this run is FOR
    run_id  TEXT             -- which Airflow run wrote it
);
"""

def main() -> int:
    with psycopg.connect(dsn(), autocommit=True) as c:
        c.execute(DDL)
        c.execute(f"INSERT INTO {SCHEMA}.heartbeat (slot, run_id) VALUES (%s, %s)",
                  (os.environ.get("SLOT", "by hand"),
                   os.environ.get("AIRFLOW_CTX_DAG_RUN_ID", "not airflow")))
    print("  heartbeat: ok  one row written")
    return 0

if __name__ == "__main__":
    sys.exit(main())
'''

(PROJECT / 'pipelines' / 'heartbeat.py').write_text(HEARTBEAT.lstrip())
print('wrote', PROJECT / 'pipelines' / 'heartbeat.py')

## Run it yourself, before Airflow is involved at all

This is the command a human types. Remember it: the DAG is going to type exactly
this and nothing else.

In [ ]:
run('-m', 'pipelines.heartbeat')

In [ ]:
from pipelines.lib.config import SCHEMA

sql(f"""
    SELECT to_char(ran_at, 'HH24:MI:SS') AS ran_at, slot, run_id
    FROM {SCHEMA}.heartbeat ORDER BY ran_at DESC LIMIT 5
""", 'one row, written by hand')

`slot` says **by hand** and `run_id` says **not airflow**, because nothing
scheduled this. Watch both of those change in a minute.

---

# Step 2 · The DAG

A DAG is **a picture of what has to happen before what**. Three words for one
idea:

| | |
|---|---|
| **D**irected | the arrows point one way |
| **A**cyclic | no loops. nothing waits for itself |
| **G**raph | a set of tasks with arrows between them |

Ours has one task, so the picture is a dot. That is fine: **the schedule is the
lesson here, not the shape.**

In [ ]:
TEACH_DAG = '''
from __future__ import annotations
import pendulum
from airflow import DAG
from airflow.operators.bash import BashOperator

# where the project is mounted INSIDE the container. Not the path on your laptop.
PROJECT = "/opt/kerb/teach"

with DAG(
    dag_id="teach_dag",
    description="the smallest useful DAG, built in notebook 12",
    schedule="*/1 * * * *",                        # every minute
    start_date=pendulum.now("UTC").subtract(minutes=5),
    catchup=False,                                 # do NOT backfill what was missed
    max_active_runs=1,                             # never two runs at once
    default_args={"retries": 1,
                  "retry_delay": pendulum.duration(minutes=1)},
    tags=["course"],
) as dag:

    BashOperator(
        task_id="heartbeat",
        # the exact command you typed above. {{ ts }} is Airflow filling in
        # which scheduled slot this run is for, at run time.
        bash_command=f"cd {PROJECT} && SLOT={{{{ ts }}}} python -m pipelines.heartbeat",
        env={"PYTHONPATH": PROJECT},
        append_env=True,
    )
'''

print(TEACH_DAG)

### Read every argument as a question being answered

| | |
|---|---|
| `schedule="*/1 * * * *"` | how often? Every minute. Normally you would write `0 * * * *` for hourly |
| `start_date` | from when does that schedule apply? |
| `catchup=False` | if I enable this today, backfill everything missed since `start_date`? **Almost always no** |
| `max_active_runs=1` | can two runs overlap? Not for a pipeline that rebuilds a window |
| `retries` | how many times before we admit it is broken? |

**`catchup` is the one that bites people.** Leave it `True` with a start date six
months ago and enabling the DAG launches several hundred runs at once, against
production, immediately.

### And notice what the task is NOT

![](img/airflow-3-task.png)

It is **not a copy of the pipeline logic.** It runs the exact command you ran
yourself two cells ago. Two things follow:

**Any Airflow failure reproduces on your laptop** by typing one line. There is
no *"it only breaks in Airflow"*.

**The pipeline has never heard of Airflow.** Swap it for something else tomorrow
and `heartbeat.py` does not change.

---

# Step 3 · Deploy it

Deploying a DAG really is this: **write the file into the folder the scheduler
watches.** No build, no registration, no restart.

In [ ]:
target = DAGS / 'teach_dag.py'
target.write_text(TEACH_DAG.lstrip())

print(f'wrote {target}')

# the scheduler rescans that folder on a timer. Ask how long that timer is.
print('rescan interval:',
      airflow('config', 'get-value', 'scheduler', 'dag_dir_list_interval', quiet=True),
      'seconds')

airflow('dags', 'reserialize', quiet=True)                # do it now, do not wait 5 minutes
airflow('dags', 'pause', 'teach_dag', quiet=True)         # arrive paused, so we can watch it start
print('deployed')

---

# Step 4 · Does Airflow see it?

Three questions, in this order. The first one is the one people skip.

In [ ]:
airflow('dags', 'list-import-errors')

`No data found` is what you want. **A DAG file that raises on import does not
become a broken DAG, it becomes no DAG at all**, and this is the only place that
is reported.

In [ ]:
out = airflow('dags', 'list', quiet=True)
print(out.splitlines()[0])
print('\n'.join(l for l in out.splitlines() if 'teach_dag' in l))

In [ ]:
airflow('tasks', 'list', 'teach_dag')

---

# Step 5 · Run it once, by hand

`airflow tasks test` runs a single task immediately, ignoring the schedule and
ignoring dependencies. It is what you use while developing.

In [ ]:
out = airflow('tasks', 'test', 'teach_dag', 'heartbeat', '2026-09-01', quiet=True)
print('\n'.join(l for l in out.splitlines()
                 if 'heartbeat:' in l or 'Running command' in l or 'Marking task' in l))

In [ ]:
sql(f"""
    SELECT to_char(ran_at, 'HH24:MI:SS') AS ran_at, slot, run_id
    FROM {SCHEMA}.heartbeat ORDER BY ran_at DESC LIMIT 5
""", 'the row Airflow just wrote')

**That ran inside the Airflow container**, against the same Postgres, using the
same file you wrote in step 1. Not a simulation.

---

# Step 6 · Turn the schedule on

Everything so far, somebody pressed. **A DAG that only runs when you press it is
a shell script with a nicer log viewer.**

First, while it is still paused, ask Airflow when it *would* run. It knows,
because a schedule is **a declared cadence**, not a process sitting in a loop.

In [ ]:
out = airflow('dags', 'next-execution', 'teach_dag', quiet=True)

# a paused DAG adds a reminder line, so pick the line that is actually a date
when = [l.strip() for l in out.splitlines() if l.strip()[:1].isdigit() and 'T' in l]
print('next slot :', when[-1] if when else out.splitlines()[-1])
print('is_paused :', [l for l in airflow('dags', 'list', quiet=True).splitlines()
                      if 'teach_dag' in l][0].split('|')[-1].strip())

It knows exactly when it would run, and it is not going to, because **nobody has
said yes yet**.

## Now say yes, and wait

In [ ]:
def runs_of(dag_id):
    out = airflow('dags', 'list-runs', '-d', dag_id, '-o', 'json', quiet=True)
    try:
        return json.loads(out)
    except json.JSONDecodeError:
        return []

already = {r['run_id'] for r in runs_of('teach_dag')}
print(f'{len(already)} runs exist already. Watching for a NEW one.\n')

airflow('dags', 'unpause', 'teach_dag', quiet=True)

t0, fresh = time.time(), []
for _ in range(24):
    fresh = [r for r in runs_of('teach_dag')
             if r['run_id'] not in already and r['run_id'].startswith('scheduled')]
    done = [r for r in fresh if r['state'] == 'success']
    print(f'{int(time.time() - t0):>4}s   new scheduled runs: {len(fresh)}, finished: {len(done)}')
    if done:
        break
    time.sleep(10)

for r in fresh:
    print(f"\n  {r['run_id']}   {r['state']}")

### Nobody triggered that

Look at the run id. It starts with **`scheduled__`**, not `manual__`. That run
exists because a cron expression in a file said it should, and the scheduler
agreed.

## And here is the row it wrote

Run this cell, talk for a minute, run it again.

In [ ]:
sql(f"""
    SELECT to_char(ran_at, 'HH24:MI:SS') AS ran_at, slot, run_id
    FROM {SCHEMA}.heartbeat ORDER BY ran_at DESC LIMIT 8
""", 'one row per scheduled minute')

### The `slot` column is the bit worth pointing at

`ran_at` is when the machine did the work. `slot` is **which minute the work was
for**. They are not the same thing, and confusing them is one of the most
expensive mistakes in this job.

A run that starts at 03:07 because the scheduler was busy is still rebuilding
the 03:00 window. Every pipeline in this course takes its window from the data
or from the slot, **never from `now()`**, for exactly that reason.

## Go and look at it

Open **http://localhost:8080** (`kerb` / `kerb_local_dev`), find `teach_dag`,
and put the **Grid** view on the screen. A new green square appears every minute
while you talk.

| In the UI | What it shows |
|---|---|
| **Grid** | one column per run, one square per task, green or red |
| **Next Run** | the next slot, the same number you printed above |
| the pause toggle | the same thing `dags pause` does |
| a square, then **Logs** | that pipeline's own output |
| **Clear** on a square | rerun just that task |

---

# Step 7 · Turn it off

An every-minute DAG left running will fill that table forever.

In [ ]:
airflow('dags', 'pause', 'teach_dag')

n_rows = fetch(f'SELECT count(*) AS n FROM {SCHEMA}.heartbeat').n[0]
print(f'\n{n_rows} heartbeat rows. That number stops moving now.')

---

## The real one, which is the same thing with more tasks

`airflow/dags/teach_pipelines.py` in this project schedules the eight pipelines
you built in notebooks 2 to 8. Same file shape, same BashOperators, same
deployment. Two extra ideas:

![](img/airflow-2-dag.png)

In [ ]:
airflow('tasks', 'list', 'teach_pipelines')

**One:** `bronze >> silver >> gold` is the whole graph. The six bronze tasks
share nothing, so Airflow runs all six at once, for free. And the guarantee that
one line gives you is absolute:

> **Gold can never be built from a silver table that failed to refresh.**

Not "should not". *Cannot.* If silver fails, Airflow does not start gold.

**Two:** the last task runs the signal board, and it **does not fail the DAG**
when a signal breaches. "A number moved and a human should look" is not the same
as "the run is broken". Failing there would page the on-call engineer for a
number that is often completely correct.

To deploy that one, the project has a command:

```bash
python cli.py deploy
```

---

## What Airflow does not do

- it does **not** move your data. Your code does
- it does **not** know whether your data is correct. That is the signal board
- it does **not** make a bad pipeline good. It runs it on time and tells you it failed

## What you learned

- Airflow **starts programs**. It never touches a row
- **Two files**: one does the work, one says when. They live in different folders
- A task runs **the same command you would type**
- **Deploying is writing a file** into a watched folder
- `catchup=False` unless you genuinely want a backfill
- A paused DAG still knows its next slot. **A schedule is a declared cadence**
- `slot` is which window the run is for. `now()` is not
- `bronze >> silver >> gold` makes stale gold **impossible**, not just unlikely